**Persiapan SparkSession**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/26 16:46:03 WARN Utils: Your hostname, xcel resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/26 16:46:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/26 16:46:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**A. Extract**

In [2]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json"
)
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


**B. Join & Kolom Turunan**

In [3]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                           .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


**C. Window Function — Top-2 Menu per Cabang**

In [4]:
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                  .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(20)

[Stage 19:>                                                         (0 + 1) / 1]

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



**D. Spark SQL — Total Penjualan per Kepala Barista**

In [5]:
df_gabungan.createOrReplaceTempView("gabungan_kn")

kontribusi_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan_kn
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')
kontribusi_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



**E. Load**

In [6]:
!hdfs dfs -mkdir -p /user/xcell/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/xcell/tugas7/processed/laporan"
)

# Verifikasi
!hdfs dfs -ls /user/xcell/tugas7/processed/laporan
df_cek = spark.read.parquet("hdfs://localhost:9000/user/xcell/tugas7/processed/laporan")
print("Jumlah baris terverifikasi:", df_cek.count())

Found 4 items
-rw-r--r--   3 xcell supergroup          0 2026-09-26 16:46 /user/xcell/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - xcell supergroup          0 2026-09-26 16:46 /user/xcell/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - xcell supergroup          0 2026-09-26 16:46 /user/xcell/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - xcell supergroup          0 2026-09-26 16:46 /user/xcell/tugas7/processed/laporan/kategori_menu=Non-Kopi
Jumlah baris terverifikasi: 4000


**F. Ringkasan Eksekutif**

In [7]:
ringkasan_eksekutif = df_gabungan.groupBy("nama_cabang").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show()

+--------------------+---------------+----------------+-------------------------+
|         nama_cabang|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|       84645000|            1305|        64862.06896551724|
+--------------------+---------------+----------------+-------------------------+



Kalau dilihat dari tabel ringkasan eksekutif, cabang Kopi Nusantara Semarang jadi yang paling unggul, dengan total penjualan Rp90.542.000 dari 1.393 transaksi, paling tinggi dibanding dua cabang lain. Di sisi lain, cabang Kopi Nusantara Yogyakarta jadi yang paling lemah performanya, dengan total penjualan Rp84.645.000 dari 1.305 transaksi. Menariknya, jumlah transaksinya sebenarnya tidak beda jauh dari cabang lain, cuma nilai rata-rata per transaksinya yang agak lebih kecil, jadi masalahnya bukan sepi pembeli, tapi nilai belanjanya yang kurang maksimal. Nah, dari hasil bagian C, dua menu paling laris di cabang Yogyakarta adalah Menu-12 dan Menu-15, jadi dua menu inilah yang cocok buat didorong lebih gencar lagi, misalnya lewat paket bundling, promo di jam-jam sepi, atau campaign kecil-kecilan di media sosial khusus cabang itu. Selain dari sisi menu, kepala barista di cabang Yogyakarta juga bisa dikasih target tambahan berdasarkan hasil bagian D, biar ada dorongan ekstra buat ningkatin penjualan. Rekomendasi ini murni berdasarkan pola data yang ada, bukan tebak-tebakan, jadi arah promosinya lebih jelas dan pas sasaran buat cabang yang bersangkutan.

In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
